# KoAlpaca로 베이스 모델 Instruction Tuning (Before / After)
**모델:** Qwen2.5-1.5B (베이스, instruct 아님)  
**데이터:** beomi/KoAlpaca-v1.1a (일부만 사용)  
**방법:** QLoRA (Unsloth)  
**환경:** Google Colab 무료 티어

> ⚠️ 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 실행하세요.

이 노트북의 목표는 *똑똑한 모델 만들기*가 아니라, **"지시를 따르는 형식(질문→답변)을 학습 전/후로 비교"** 하는 것입니다. 앞서 한 BERT 분류 실습과 짝을 이루는 생성 실습용입니다.

## 1. 패키지 설치

In [ ]:
!pip install -q --upgrade datasets
!pip install -q unsloth

지저분한 경고 메세지를 띄우지 않기 위한 코드

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import transformers
transformers.logging.set_verbosity_error()

## 2. 베이스 모델 로드
**instruct가 아닌 베이스 모델**을 불러옵니다. 베이스 모델은 아직 "질문에 답하는 법"을 배우지 않은 상태라, 학습 전/후 차이가 분명하게 드러납니다. (조선시대 노트북과 비교하면 모델 이름에서 `-Instruct`만 빠졌습니다.)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B",
    max_seq_length = 1024,
    load_in_4bit = True,
)
print("✅ 베이스 모델 로드 완료 (instruct 아님)")

## 3. LoRA 설정
조선시대 노트북과 동일한 설정입니다. LoRA를 막 붙인 직후에는 **변화량이 0으로 초기화**되어 있어서, 바로 아래 "학습 전" 출력은 사실상 베이스 모델 그대로의 결과입니다.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)
print("✅ LoRA 설정 완료")

## 4. 학습 전 결과 (Before)
학습하기 전에 같은 질문들을 먼저 던져봅니다. 베이스 모델이라 다음과 같은 증상이 보일 수 있습니다:
- 질문에 답하지 않고 비슷한 질문을 계속 만들어냄
- 답을 끝맺지 못하고 장황하게 이어짐
- 중간에 영어/중국어가 섞임

이 출력을 기억해두고, 학습 후와 비교합니다. (학습 데이터와 **똑같은 `### 질문 / ### 답변` 형식**으로 물어봅니다 — 형식을 맞춰주는 게 중요합니다.)

In [ ]:
test_questions = [
    "양파는 어떤 식물의 부위인가요?",
    "건강을 유지하는 방법을 알려주세요.",
    "세종대왕의 업적을 설명해주세요.",
]

def make_prompt(q):
    return f"""### 질문: {q}

    ### 답변:"""

def generate(q, max_new_tokens=200):
    FastLanguageModel.for_inference(model)
    inputs = tokenizer(make_prompt(q), return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        do_sample=True,
    )
    gen = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return gen.strip()

before_answers = {}
for q in test_questions:
    before_answers[q] = generate(q)
    print(f"Q: {q}")
    print(f"A: {before_answers[q]}")
    print("-" * 50)

## 5. 데이터 준비

> **"데이터를 일부만 쓰면 성능이 안 나오지 않나요?"**
>
> 이 실습에서 가르치는 건 *지식*이 아니라 **형식(질문 → 한국어로 답하고 멈추기)** 입니다. 형식은 적은 데이터로도 빠르게 학습됩니다. 그래서 KoAlpaca 2만여 개 중 **1,500개만** 써도 before/after 차이는 충분히 보입니다.
>
> 대신 *얻지 못하는 것*도 분명합니다: 광범위한 사실 정확도나 추론 능력. 그건 1.5B 모델 + 소량 데이터의 목표가 아닙니다. **"loss가 떨어졌는가(학습 건강)"** 와 **"출력이 쓸만한가(직접 읽어 판단)"** 는 별개 축이라는 점을 여기서 확인하세요.

답변이 너무 길거나 지저분한 샘플은 걸러내고, 짧고 깔끔한 것들만 골라 학습 속도와 품질을 높입니다.

In [ ]:
from datasets import load_dataset

ds = load_dataset("beomi/KoAlpaca-v1.1a", split="train")

def is_clean(ex):
    ins, out = ex["instruction"], ex["output"]
    return bool(ins) and bool(out) and len(ins) <= 150 and len(out) <= 300

N_SAMPLES = 1500
ds = ds.filter(is_clean).shuffle(seed=42)
n = min(N_SAMPLES, len(ds))
ds = ds.select(range(n))

def format_example(ex):
    text = f"""### 질문: {ex['instruction']}

### 답변: {ex['output']}""" + tokenizer.eos_token
    return {"text": text}

dataset = ds.map(format_example, remove_columns=ds.column_names)
print(f"✅ 데이터셋 준비 완료: {len(dataset)}개")
print("--- 샘플 확인 ---")
print(dataset[0]["text"])

## 6. 학습 실행
T4 기준 약 **2~4분** 소요됩니다 (데이터 1,500개 × 2 epoch).
빠르게 끝나면 `N_SAMPLES`나 `num_train_epochs`를 늘리고, 너무 오래 걸리면 줄이세요.

In [ ]:
from trl import SFTTrainer, SFTConfig

FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = 1024,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        warmup_ratio = 0.1,
        lr_scheduler_type = "cosine",
        fp16 = True,
        logging_steps = 10,
        output_dir = "./koalpaca_output",
        report_to = "none",
    ),
)

trainer.train()
print("✅ 학습 완료")

## 7. 학습 후 결과 + Before / After 비교
같은 질문을 다시 던지고, 학습 전 답변과 나란히 비교합니다. **형식이 잡히고(질문에 답하고 멈춤), 한국어로 끝맺는지**를 보세요. 사실관계가 완벽한지가 아니라 *행동이 바뀌었는지*가 포인트입니다.

In [ ]:
after_answers = {}
for q in test_questions:
    after_answers[q] = generate(q)

for q in test_questions:
    print(f"❓ {q}")
    print(f"[Before] {before_answers[q]}")
    print(f"[After ] {after_answers[q]}")
    print("=" * 60)

## 8. 모델 저장 (선택)
LoRA 가중치만 저장합니다 (용량 작음).

In [ ]:
model.save_pretrained("koalpaca_base_lora")
tokenizer.save_pretrained("koalpaca_base_lora")

# 구글 드라이브에 저장하려면
# from google.colab import drive
# drive.mount('/content/drive')
# model.save_pretrained("/content/drive/MyDrive/koalpaca_base_lora")

print("✅ 저장 완료 → koalpaca_base_lora/")